In [ ]:
import matplotlib.pyplot as plt
from os.path import join
import sys
import numpy as np
import matplotlib.pyplot as plt

LOAD_DIR = 'modified_swiss_dwellings/'
MAX_ITER = 20_000
ABS_TOL  = 1e-4

def load_data(load_dir, bid):
    SIZE = 512
    u = np.zeros((SIZE + 2, SIZE + 2))
    u[1:-1, 1:-1] = np.load(join(load_dir, f"{bid}_domain.npy"))
    interior_mask = np.load(join(load_dir, f"{bid}_interior.npy"))
    return u, interior_mask

def visualize(bid):
    print(f"Loading building {bid} ...")
    u_before, mask = load_data(LOAD_DIR, bid)


    domain_before = u_before[1:-1, 1:-1]
    wall_mask    = (~mask) & (domain_before != 0)
    interior = mask.astype(int)


    plot_dmoain = domain_before
    plot_interior  = mask
    plot_interior = mask*25

    vmin = 0
    vmax = np.nanmax(plot_dmoain)
    print(np.max(domain_before))
    print(np.max(mask))

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    cmap = plt.cm.RdBu_r.copy()
    cmap.set_bad(color='lightgrey')

    im = ax[0].imshow(plot_dmoain, origin='upper', cmap=cmap,
                       vmin=vmin, vmax=vmax, interpolation='nearest')
    ax[0].set_title(f"Domain of floorplan {bid}", fontsize=16)
    ax[0].axis('off')
    fig.colorbar(im, ax=ax[0], fraction=0.046, pad=0.04, label='Temperature (°C)')


    im = ax[1].imshow(plot_interior, origin='upper', cmap='gray',
                       vmin=vmin, vmax=vmax, interpolation='nearest')
    ax[1].set_title(f"Interior of floorplan {bid}", fontsize=16)
    ax[1].axis('off')
    

    plt.tight_layout()
    out = f"visualizations/building_{bid}_domain_and_interior.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    print(f"Saved to {out}")
    plt.show()


visualize('23')

In [ ]:

# ── Speedup analysis from results.out (N=20 floorplans) ──────────────────────

import matplotlib.pyplot as plt
import numpy as np

baseline = 168.55  # seconds (1 worker, simulate_OG.py)

workers = list(range(2, 17))

static_times = [
    97.34, 69.77, 53.16, 40.86, 40.53,
    36.34, 35.02, 35.16, 28.68, 29.89,
    29.66, 30.99, 29.94, 28.26, 30.51,
]

dynamic_times = [
    87.08, 62.59, 53.78, 40.53, 38.25,
    33.06, 31.99, 29.46, 28.16, 27.46,
    26.66, 25.46, 24.46, 23.68, 24.08,
]

static_speedup  = [baseline / t for t in static_times]
dynamic_speedup = [baseline / t for t in dynamic_times]
ideal_speedup   = [w for w in workers]  # perfect linear scaling

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(workers, static_speedup,  marker='o', label='Static scheduling',  color='steelblue')
ax.plot(workers, dynamic_speedup, marker='s', label='Dynamic scheduling', color='darkorange')
ax.plot(workers, ideal_speedup,   linestyle='--', color='gray', label='Ideal (linear)')

ax.set_xlabel('Number of workers', fontsize=13)
ax.set_ylabel('Speedup  $S = T_1 / T_p$', fontsize=13)
ax.set_title('Parallel speedup vs. number of workers\n(N=20 floorplans, baseline = {:.2f} s)'.format(baseline), fontsize=13)
ax.set_xticks(workers)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/speedup_static_vs_dynamic.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"{'Workers':>8} | {'Static speedup':>15} | {'Dynamic speedup':>16}")
print("-" * 46)
for w, ss, ds in zip(workers, static_speedup, dynamic_speedup):
    print(f"{w:>8} | {ss:>15.2f} | {ds:>16.2f}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
result_df = pd.read_csv('full_results.csv')
display(result_df)

In [ ]:
clean_df = result_df.dropna(subset=[' mean_temp'])

fig, axes = plt.subplots(1, 1, figsize=(8, 5))
fig.suptitle('Temperature Across Floorplans', fontsize=15, fontweight='bold')

axes.hist(clean_df[' mean_temp'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes.set_title('Mean Temperature', fontsize=13)
axes.set_xlabel('Temperature (°C)')
axes.set_ylabel('Count')
axes.axvline(clean_df[' mean_temp'].mean(), color='red', linestyle='--', label=f"Mean: {clean_df[' mean_temp'].mean():.1f}°C")
axes.legend()


plt.tight_layout()
plt.savefig('visualizations/temperature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()